# Ch 03 · Rigid-Body Physics from Scratch: Particles, Rigid Bodies, Joints and Contacts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/03-rigid-body-physics-primer.ipynb)

Companion notebook for the chapter page `chapters/03-rigid-body-physics-primer.html` ([page](../chapters/03-rigid-body-physics-primer.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## How to use this notebook
This chapter builds rigid-body physics from scratch. Examples 3.1, 3.2, 3.5 and 3.8 are plain NumPy or C MuJoCo and run on any machine; 3.3, 3.4, 3.6, 3.7, 3.9 and 3.10 also run MJWarp. Every example cell is the same file as on the chapter page. Run the cells top to bottom; each one is standalone.

## §1 A point mass: F = m a and time stepping
A point mass has state (x, v). Each time step computes the force, a = F / m, and writes the next state. The two loops below differ only in the ORDER of the two update lines.

In [ ]:
# Example 3.1 - ex1_point_mass.py
"""Example 3.1 - a point mass under gravity: explicit vs semi-implicit Euler, in plain NumPy.

Both methods take the same 200 steps of dt = 0.01 s. Only the ORDER of the two updates differs:
  explicit Euler:       x += v * dt;  v += a * dt     (position uses the OLD velocity)
  semi-implicit Euler:  v += a * dt;  x += v * dt     (position uses the NEW velocity, like MuJoCo)
Run: python ex1_point_mass.py
"""
import numpy as np

np.set_printoptions(precision=4, suppress=True)

G = 9.81          # gravity [m/s^2], pointing down (-z)
M = 1.0           # mass [kg]
Z0, V0 = 20.0, 0.0
DT, T = 0.01, 2.0
NSTEP = round(T / DT)


def energy(z, v):
    """Total mechanical energy = kinetic + potential [J]."""
    return 0.5 * M * v**2 + M * G * z


def explicit_euler(z, v):
    for _ in range(NSTEP):
        a = -G                 # F = m a  ->  a = F / m = -m g / m
        z, v = z + v * DT, v + a * DT     # both right-hand sides use the old (z, v)
    return z, v


def semi_implicit_euler(z, v):
    for _ in range(NSTEP):
        a = -G
        v = v + a * DT         # 1. velocity first ...
        z = z + v * DT         # 2. ... then position with the NEW velocity
    return z, v


z_exact = Z0 + V0 * T - 0.5 * G * T**2
v_exact = V0 - G * T
E0 = energy(Z0, V0)
print(f"{NSTEP} steps of dt = {DT} s, t = {T} s, z0 = {Z0} m")
print(f"analytic      : z = {z_exact:.4f} m   v = {v_exact:.4f} m/s")
for name, fn in [("explicit", explicit_euler), ("semi-implicit", semi_implicit_euler)]:
    z, v = fn(Z0, V0)
    print(f"{name:<14}: z = {z:.4f} m   v = {v:.4f} m/s   "
          f"z error = {z - z_exact:+.4f} m   energy error = {energy(z, v) - E0:+.4f} J")
print(f"predicted |z error| = g*T*dt/2 = {G * T * DT / 2:.4f} m (same size, opposite sign)")

## §2 Explicit vs semi-implicit Euler
On a spring (a = -omega^2 x), explicit Euler multiplies the energy by (1 + omega^2 dt^2) every step. Semi-implicit Euler (velocity first, MuJoCo's `euler`: `_next_velocity` then `_next_position`) keeps it bounded while omega*dt < 2.

In [ ]:
# Example 3.2 - ex2_spring_oscillator.py
"""Example 3.2 - a 1-D mass-spring: explicit Euler gains energy, semi-implicit Euler stays bounded.

The spring has period 1 s (omega = 2 pi). For dt in {0.01, 0.05, 0.2} both methods run 10 s and we
print the largest energy ratio E(t) / E(0). Exact physics keeps the ratio at 1.
Run: python ex2_spring_oscillator.py [--dump-json PATH]
  --dump-json PATH writes (x, v) phase-portrait traces for the page's figure (Rule D).
"""
import argparse
import json

import numpy as np

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()   # parse_known_args: also works inside Jupyter

M = 1.0
K = (2 * np.pi) ** 2      # stiffness so that omega = sqrt(K / M) = 2 pi rad/s (period 1 s)
X0, V0 = 1.0, 0.0
T = 10.0
DTS = [0.01, 0.05, 0.2]


def energy(x, v):
    return 0.5 * M * v**2 + 0.5 * K * x**2


def simulate(dt, semi_implicit):
    x, v = X0, V0
    xs, vs = [x], [v]
    for _ in range(round(T / dt)):
        a = -K * x / M                    # Hooke's law F = -k x, then a = F / m
        if semi_implicit:
            v = v + a * dt
            x = x + v * dt
        else:
            x, v = x + v * dt, v + a * dt
        xs.append(x)
        vs.append(v)
    return np.array(xs), np.array(vs)


E0 = energy(X0, V0)
dump = {"omega": 2 * np.pi, "runs": []}
print(f"omega = {np.sqrt(K / M):.4f} rad/s, T = {T} s, E0 = {E0:.4f} J")
print(f"{'dt':>6} {'omega*dt':>9} {'explicit max E/E0':>18} {'semi-implicit max E/E0':>23}")
for dt in DTS:
    ex = simulate(dt, semi_implicit=False)
    si = simulate(dt, semi_implicit=True)
    r_ex = energy(*ex).max() / E0
    r_si = energy(*si).max() / E0
    print(f"{dt:>6} {np.sqrt(K / M) * dt:>9.3f} {r_ex:>18.4g} {r_si:>23.4f}")
    # keep the first 3 s for the figure; at most ~150 points per trace
    n = round(3.0 / dt) + 1
    step = max(1, n // 150)
    dump["runs"].append({
        "dt": dt,
        "explicit": [[round(float(a), 3), round(float(b), 3)]
                     for a, b in zip(ex[0][:n:step], ex[1][:n:step])],
        "semi": [[round(float(a), 3), round(float(b), 3)]
                 for a, b in zip(si[0][:n:step], si[1][:n:step])],
    })

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

### Interactive phase portraits
Move the slider (Colab has ipywidgets) to change dt and watch explicit Euler spiral out while semi-implicit Euler stays on a closed loop. Past omega*dt = 2 (dt > 0.32; the slider goes to 0.4) semi-implicit Euler explodes too.

In [ ]:
# Interactive phase portraits: drag the slider (ipywidgets) or call show(dt) yourself.
import numpy as np
try:
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("skipped: matplotlib not installed")


def euler_paths(dt, T=3.0, omega=2 * np.pi):
    paths = {}
    for name, semi in [("explicit", False), ("semi-implicit", True)]:
        x, v, pts = 1.0, 0.0, [(1.0, 0.0)]
        for _ in range(round(T / dt)):
            a = -omega**2 * x
            if semi:
                v += a * dt
                x += v * dt
            else:
                x, v = x + v * dt, v + a * dt
            pts.append((x, v / omega))
        paths[name] = np.array(pts)
    return paths


def show(dt=0.05):
    paths = euler_paths(dt)
    for name, p in paths.items():
        print(f"dt = {dt}: {name:<13} final E/E0 = {(p[-1] ** 2).sum():.4g}")
    if plt is None:
        return
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(th), np.sin(th), "k--", lw=1, label="exact")
    ax.plot(*paths["explicit"].T, "r.-", label="explicit")
    ax.plot(*paths["semi-implicit"].T, "g.-", label="semi-implicit")
    ax.set(xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), xlabel="x", ylabel="v / omega", aspect="equal",
           title=f"dt = {dt} s, omega*dt = {2 * np.pi * dt:.2f}")
    ax.legend(loc="upper right")
    plt.show()


try:
    from ipywidgets import FloatLogSlider, interact
    interact(show, dt=FloatLogSlider(value=0.05, base=10, min=-2.3, max=-0.4, step=0.05))
except ImportError:
    for dt in (0.01, 0.05, 0.2):
        show(dt)

## §3 A bouncing ball on a spring-damper floor
solref (timeconst, dampratio) maps to k/m = 1/(timeconst*dampratio)^2 and b/m = 2/timeconst (`constraint.py::_contact_kbimp`). dampratio is the damping ratio: 1 means no bounce. The same drop then runs in C MuJoCo and MJWarp.

In [ ]:
# Example 3.3 - ex3_bouncing_ball_numpy.py
"""Example 3.3 - a ball on a penalty-spring floor in NumPy, next to the same drop in MuJoCo/MJWarp.

The NumPy floor is a spring-damper that pushes up only while the ball is below the floor. Its
stiffness k and damping b come from a time constant and a damping ratio with the same formula
MuJoCo uses for solref (with dmax = solimp[1] set to 1):  k/m = 1 / (timeconst * dampratio)^2,
b/m = 2 / timeconst. MuJoCo's default solref is (0.02, 1).
Run: python ex3_bouncing_ball_numpy.py [--dump-json PATH]
  --dump-json PATH writes the height traces for the page's figure (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

G, R, Z0 = 9.81, 0.1, 1.0     # gravity, ball radius, start height of the centre
DT, T = 0.002, 1.5
NSTEP = round(T / DT)


def numpy_ball(timeconst, dampratio):
    """Semi-implicit Euler with a one-sided spring-damper floor. Returns heights and penetration."""
    k = 1.0 / (timeconst * dampratio) ** 2       # per unit mass [1/s^2]
    b = 2.0 / timeconst                          # per unit mass [1/s]
    z, v = Z0, 0.0
    zs = [z]
    for _ in range(NSTEP):
        pen = R - z                              # > 0 when the ball is inside the floor
        a = -G
        if pen > 0:
            a += max(0.0, k * pen - b * v)       # push only, never pull (one-sided)
        v += a * DT
        z += v * DT
        zs.append(z)
    return np.array(zs)


def bounce_peaks(zs):
    """Heights of the local maxima after the first impact (bounce apex of the centre)."""
    i0 = int(np.argmax(zs < R + 1e-3))
    peaks = [zs[i] for i in range(i0 + 1, len(zs) - 1) if zs[i - 1] < zs[i] >= zs[i + 1]]
    return [round(float(p), 3) for p in peaks if p > R + 1e-3][:4]


print(f"NumPy penalty floor, dt = {DT} s, {NSTEP} steps, ball radius {R} m, drop from {Z0} m")
traces = {}
for tc, dr in [(0.02, 1.0), (0.02, 0.3), (0.02, 0.15)]:
    zs = numpy_ball(tc, dr)
    traces[f"{tc}_{dr}"] = zs
    print(f"  solref ({tc}, {dr:<4}): k/m = {1 / (tc * dr) ** 2:6.0f} 1/s^2  "
          f"b/m = {2 / tc:.0f} 1/s  "
          f"max penetration = {1000 * (R - zs.min()):5.2f} mm  "
          f"bounce apexes [m] = {bounce_peaks(zs)}")

# --- the same drop in C MuJoCo (default solref 0.02, 1) ---
mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
mjd = mujoco.MjData(mjm)
z_c, pen_c = [], 0.0
for _ in range(NSTEP):
    mujoco.mj_step(mjm, mjd)
    z_c.append(mjd.qpos[2])
    if mjd.ncon:
        pen_c = max(pen_c, -mjd.contact.dist.min())

# --- and in MJWarp, one world ---
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=4, njmax=16)
z_w, pen_w = [], 0.0
for _ in range(NSTEP):
    mjw.step(m, d)
    z_w.append(float(d.qpos.numpy()[0, 2]))        # reads to host every step: demo only
    n = int(d.nacon.numpy()[0])
    if n:
        pen_w = max(pen_w, -float(d.contact.dist.numpy()[:n].min()))

zn = traces["0.02_1.0"]
print("max penetration, same drop, default solref (0.02, 1):")
print(f"  NumPy penalty floor : {1000 * (R - zn.min()):.2f} mm   rest {1000 * (R - zn[-1]):.2f} mm")
print(f"  C MuJoCo            : {1000 * pen_c:.2f} mm   rest {1000 * (R - z_c[-1]):.2f} mm")
print(f"  MJWarp (1 world)    : {1000 * pen_w:.2f} mm   rest {1000 * (R - z_w[-1]):.2f} mm")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    every = 10          # 0.02 s per frame
    out = {"dt": DT * every, "radius": R,
           "numpy": {k: [round(float(z), 4) for z in v[::every]] for k, v in traces.items()},
           "mujoco": [round(float(z), 4) for z in z_c[::every]],
           "mjwarp": [round(float(z), 4) for z in z_w[::every]]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## §4 A rigid body: 6 DOF, qpos 7 and qvel 6
Position (3) + unit quaternion (w, x, y, z) (4) + linear velocity (3, world frame) + angular velocity (3, BODY frame).

In [ ]:
# Example 3.4 - ex4_rigid_body_state.py
"""Example 3.4 - the state of one free rigid body: 7 numbers of position, 6 of velocity.

A free box of full side lengths 0.2 x 0.4 x 0.6 m and mass 3 kg. We print nq, nv, the unit
quaternion qpos[3:7], the angular velocity qvel[3:6], and the mass properties MuJoCo computed,
and check body_inertia against the textbook box formula I_xx = m (b^2 + c^2) / 12.
Run: python ex4_rigid_body_state.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

FREE_BOX = """
<mujoco model="free_box">
  <option timestep="0.002" gravity="0 0 0"/>
  <worldbody>
    <body name="box" pos="0 0 1">
      <freejoint/>
      <geom type="box" size=".1 .2 .3" mass="3"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(FREE_BOX)
mjd = mujoco.MjData(mjm)
mjd.qvel[:] = [0.5, 0.0, 0.0, 0.0, 0.0, 2.0]      # 0.5 m/s along x, 2 rad/s about the body z axis
for _ in range(500):                              # 1 s
    mujoco.mj_step(mjm, mjd)

print("nq =", mjm.nq, " nv =", mjm.nv, " (free joint: 3 position + 4 quaternion, 3 + 3 velocity)")
print("qpos[0:3] position   :", mjd.qpos[0:3])
print("qpos[3:7] quaternion :", mjd.qpos[3:7],
      " |q| =", round(float(np.linalg.norm(mjd.qpos[3:7])), 6))
print("qvel[0:3] linear vel :", mjd.qvel[0:3])
print("qvel[3:6] angular vel:", mjd.qvel[3:6])
angle = 2 * np.arccos(mjd.qpos[3])
print(f"rotation angle from quat = {angle:.4f} rad (expected 2 rad/s * 1 s = 2.0)")

body = 1                                           # body 0 is the world
a, b, c = 0.2, 0.4, 0.6                           # full side lengths
m_box = 3.0
I_formula = m_box / 12 * np.array([b**2 + c**2, a**2 + c**2, a**2 + b**2])
print("body_mass   :", mjm.body_mass[body])
print("body_inertia:", mjm.body_inertia[body], " (principal moments, diagonal)")
print("formula     :", I_formula)
print("body_iquat  :", mjm.body_iquat[body], " (principal axes = body axes here)")
print("body_ipos   :", mjm.body_ipos[body], " (centre of mass in the body frame)")

# The same constants on the device: mjw.put_model copies them into Warp arrays.
# Shape (1, nbody) of vec3: the leading 1 is a batch axis that domain randomization grows (Ch 10).
m = mjw.put_model(mjm)
print("MJWarp m.body_inertia: shape", m.body_inertia.shape, "dtype", m.body_inertia.dtype.__name__,
      "| body 1:", m.body_inertia.numpy()[0, body])

## §5 Mass properties: com, inertia tensor, principal axes
MuJoCo stores the principal moments (`body_inertia`) and the rotation to the principal axes (`body_iquat`), never the full 3x3 tensor. Rebuild it with R diag(I) R^T.

In [ ]:
# Example 3.5 - ex5_inertia_tensor.py
"""Example 3.5 - the inertia tensor of an L-shaped body, by hand, and what MuJoCo stores.

Two boxes form an "L" in the x-z plane. We build the full 3x3 inertia tensor about the combined
centre of mass with the parallel-axis theorem, diagonalize it with NumPy, and compare with
MuJoCo's body_ipos (centre of mass), body_inertia (principal moments) and body_iquat (rotation
from the body frame to the principal axes).
Run: python ex5_inertia_tensor.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

L_SHAPE = """
<mujoco model="l_shape">
  <worldbody>
    <body name="L" pos="0 0 1">
      <freejoint/>
      <geom name="arm" type="box" pos=".3 0 0"   size=".3 .05 .05" mass="2"/>
      <geom name="leg" type="box" pos=".05 0 .25" size=".05 .05 .2" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

# (mass, centre in the body frame, half sizes) of each box, copied from the XML above
parts = [(2.0, np.array([0.3, 0.0, 0.0]), np.array([0.3, 0.05, 0.05])),
         (1.0, np.array([0.05, 0.0, 0.25]), np.array([0.05, 0.05, 0.2]))]

mass = sum(p[0] for p in parts)
com = sum(p[0] * p[1] for p in parts) / mass             # mass-weighted mean position

I = np.zeros((3, 3))
for m_i, c_i, h in parts:
    full = 2 * h                                           # full side lengths
    I_own = m_i / 12 * np.diag([full[1]**2 + full[2]**2,   # box about its own centre
                                full[0]**2 + full[2]**2,
                                full[0]**2 + full[1]**2])
    r = c_i - com                                          # parallel-axis theorem:
    I += I_own + m_i * (r @ r * np.eye(3) - np.outer(r, r))  # I_com = I_own + m (|r|^2 E - r r^T)

evals, evecs = np.linalg.eigh(I)                           # symmetric -> real eigen-decomposition
print("mass:", mass, " centre of mass:", com)
print("full inertia tensor about the com (note the off-diagonal I_xz):")
print(I)
print("NumPy principal moments (ascending):", evals)

mjm = mujoco.MjModel.from_xml_string(L_SHAPE)
b = mjm.body("L").id
R = np.zeros(9)
mujoco.mju_quat2Mat(R, mjm.body_iquat[b])
R = R.reshape(3, 3)                                        # columns = principal axes in body frame
print("MuJoCo body_ipos    :", mjm.body_ipos[b])
print("MuJoCo body_inertia :", mjm.body_inertia[b])
print("MuJoCo body_iquat   :", mjm.body_iquat[b])
print("sorted moments match:", np.allclose(np.sort(mjm.body_inertia[b]), evals, atol=1e-6))
I_back = R @ np.diag(mjm.body_inertia[b]) @ R.T              # rotate the diagonal back
print("R diag(body_inertia) R^T equals the hand-built tensor:", np.allclose(I_back, I, atol=1e-6))
# the axis of the smallest moment is the "easy" spin axis; where does it point in the x-z plane?
v_np = evecs[:, 0]
v_mj = R[:, int(np.argmin(mjm.body_inertia[b]))]
ang = lambda v: (np.degrees(np.arctan2(v[2], v[0])) + 90) % 180 - 90   # axis: sign does not matter
print(f"easiest spin axis is tilted from +x by: NumPy {ang(v_np):.2f} deg, "
      f"MuJoCo {ang(v_mj):.2f} deg")

## §6 Integrating an orientation
The exact update is q <- q * exp(omega dt / 2) (`math.quat_integrate`, called by `_next_position` for free and ball joints). Add-and-renormalize falls behind by about (|omega| dt)^3 / 12 per step. This example imports `mujoco_warp._src.math`: private API, the name may change.

In [ ]:
# Example 3.6 - ex6_quat_integrate.py
"""Example 3.6 - integrating an orientation: the quaternion exponential vs "add and renormalize".

A body spins with a constant angular velocity w (body frame, 10 rad/s) for 1000 steps of 0.01 s.
  exact update : q <- q * exp(w dt / 2)              (what MuJoCo and MJWarp do)
  naive update : q <- q + dt * 0.5 * q * (0, w), then divide by |q|
We print the angle error of each against the closed-form answer, and the norm of the naive
quaternion before renormalization. Then we run MJWarp's own math.quat_integrate (private API)
inside a small Warp kernel, and C MuJoCo's mju_quatIntegrate, and check they agree.
Run: python ex6_quat_integrate.py [--dump-json PATH]
  --dump-json PATH writes the angle error per step for the page's chart (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
from mujoco_warp._src import math as mjw_math      # private API: the name may change

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

DT, NSTEP = 0.01, 1000
W = np.array([2.0, -3.0, 9.0])
W = 10.0 * W / np.linalg.norm(W)                     # |w| = 10 rad/s about a tilted axis
Q0 = np.array([np.cos(0.4), np.sin(0.4), 0.0, 0.0])  # start rotated 0.8 rad about x


def qmul(a, b):
    """Hamilton product of (w, x, y, z) quaternions (same formula as math.mul_quat)."""
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b
    return np.array([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
                     w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
                     w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2])


def qexp(rotvec):
    """Unit quaternion of a rotation by |rotvec| about rotvec / |rotvec|."""
    angle = np.linalg.norm(rotvec)
    if angle < 1e-12:
        return np.array([1.0, 0.0, 0.0, 0.0])
    return np.concatenate([[np.cos(angle / 2)], np.sin(angle / 2) * rotvec / angle])


def angle_between(a, b):
    """Rotation angle that takes a to b (q and -q are the same rotation, hence abs)."""
    rel = qmul(a * np.array([1.0, -1.0, -1.0, -1.0]), b) / np.linalg.norm(a) / np.linalg.norm(b)
    return 2 * np.arctan2(np.linalg.norm(rel[1:]), abs(rel[0]))


q_exact, q_naive, q_raw = Q0.copy(), Q0.copy(), Q0.copy()
err_exact, err_naive, norms = [], [], []
for n in range(1, NSTEP + 1):
    q_exact = qmul(q_exact, qexp(W * DT))                       # exponential map
    q_naive = q_naive + DT * 0.5 * qmul(q_naive, np.r_[0.0, W])  # first-order derivative step
    norms.append(np.linalg.norm(q_naive))
    q_naive = q_naive / np.linalg.norm(q_naive)                 # renormalize
    q_raw = q_raw + DT * 0.5 * qmul(q_raw, np.r_[0.0, W])       # same, never renormalized
    q_true = qmul(Q0, qexp(W * DT * n))                         # closed form for constant w
    err_exact.append(angle_between(q_exact, q_true))
    err_naive.append(angle_between(q_naive, q_true))

print(f"|w| = {np.linalg.norm(W):.1f} rad/s, dt = {DT}, {NSTEP} steps (t = {NSTEP * DT:.0f} s, "
      f"{np.linalg.norm(W) * NSTEP * DT / (2 * np.pi):.1f} turns)")
print(f"exact exp update  : final angle error = {err_exact[-1]:.2e} rad")
print(f"naive + normalize : final angle error = {err_naive[-1]:.4f} rad "
      f"({np.degrees(err_naive[-1]):.2f} deg)")
print(f"naive |q| before each renormalization = {norms[0]:.6f} (sqrt(1 + (|w| dt / 2)^2) = "
      f"{np.sqrt(1 + (np.linalg.norm(W) * DT / 2) ** 2):.6f})")
print(f"naive without renormalizing: |q| = {np.linalg.norm(q_raw):.3f} after {NSTEP} steps")


@wp.kernel
def spin_kernel(q0: wp.array(dtype=wp.quat), w: wp.vec3, dt: float, nstep: int,
                q_out: wp.array(dtype=wp.quat)):
    q = q0[0]                              # MJWarp keeps (w, x, y, z) in a wp.quat (Ch 4)
    for _ in range(nstep):
        q = mjw_math.quat_integrate(q, w, dt)
    q_out[0] = q


q_in = wp.array([wp.quat(*Q0.astype(np.float32))], dtype=wp.quat)
q_out = wp.zeros(1, dtype=wp.quat)
wp.launch(spin_kernel, dim=1, inputs=[q_in, wp.vec3(*W.astype(np.float32)), DT, NSTEP],
          outputs=[q_out])
q_mjw = q_out.numpy()[0].astype(np.float64)

q_c = Q0.copy()
for _ in range(NSTEP):
    mujoco.mju_quatIntegrate(q_c, W, DT)   # C MuJoCo: q <- q * exp(W * DT / 2)

print("closed form       :", qmul(Q0, qexp(W * DT * NSTEP)))
print("MJWarp float32    :", q_mjw, f" error {angle_between(q_mjw, q_true):.1e} rad")
print("C MuJoCo float64  :", q_c, f" error {angle_between(q_c, q_true):.1e} rad")

if args.dump_json:
    idx = list(range(9, NSTEP, 10))
    with open(args.dump_json, "w") as f:
        json.dump({"dt": DT, "step": [i + 1 for i in idx],
                   "exact": [float(f"{err_exact[i]:.3e}") for i in idx],
                   "naive": [round(err_naive[i], 5) for i in idx],
                   "mjwarp_final": float(f"{angle_between(q_mjw, q_true):.2e}"),
                   "mujoco_final": float(f"{angle_between(q_c, q_true):.2e}")},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

### The tennis racket in many worlds
Euler's rotation equation I dw/dt + w x (I w) = tau makes spin about the intermediate axis unstable. Each world starts with a different tiny wobble; smaller wobbles flip later. On a GPU this runs 4096 worlds in the same `mjw.step` call and prints 8 of them.

In [ ]:
# Example 3.7 - ex7_tennis_racket.py
"""Example 3.7 - the tennis-racket (intermediate-axis) flip, in many MJWarp worlds at once.

A free box with three different principal moments spins at 6 rad/s about its INTERMEDIATE axis
(body y), with gravity off. Each world adds a different tiny wobble about body x. The wobble grows
and the box flips over, again and again; worlds with a smaller wobble flip later. The model uses
the RK4 integrator, so kinetic energy stays constant to float32 round-off; a C MuJoCo run of
world 0 with the default Euler integrator shows the first-order drift you would get instead (Ch 13).
We also check that MJWarp's bias force for the rotational dofs is the gyroscopic term w x (I w).
Run: python ex7_tennis_racket.py [--dump-json PATH]
  --dump-json PATH writes per-world quaternions for the page's replay player (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

RACKET = """
<mujoco model="tennis_racket">
  <option timestep="0.005" gravity="0 0 0" integrator="RK4"/>
  <worldbody>
    <body name="racket" pos="0 0 1">
      <freejoint/>
      <geom type="box" size=".05 .2 .4" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 800                                   # 4 s at dt = 0.005 (same on CPU and GPU)
SPIN = 6.0                                    # rad/s about the intermediate axis

mjm = mujoco.MjModel.from_xml_string(RACKET)
I = mjm.body_inertia[1]
print("principal moments I =", I, "-> body y is the intermediate axis:", I[2] < I[1] < I[0])
lam = SPIN * np.sqrt((I[1] - I[2]) * (I[0] - I[1]) / (I[0] * I[2]))
print(f"linear theory: the wobble grows like exp({lam:.2f} t)")

m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

wobble = SPIN * np.logspace(-4, -1.5, NWORLD)          # per-world wobble about body x [rad/s]
qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
qvel[:, 3] = wobble                     # qvel[3:6] is the BODY-frame angular velocity
qvel[:, 4] = SPIN
d.qvel.assign(qvel)


def kinetic_energy(qv):
    w = qv[:, 3:6]
    return 0.5 * (w**2 * I).sum(axis=1)            # principal axes = body axes here (iquat = 1)


ke0 = kinetic_energy(qvel)
flip_t = np.full(NWORLD, np.nan)
show = np.linspace(0, NWORLD - 1, 8).astype(int)   # the 8 worlds the page replays
frames = []
for i in range(NSTEP):
    mjw.step(m, d)
    y_axis_y = d.xmat.numpy()[:, 1, 1, 1]           # world-y part of the body y axis (demo read)
    new = np.isnan(flip_t) & (y_axis_y < 0)
    flip_t[new] = (i + 1) * mjm.opt.timestep
    if args.dump_json and i % 4 == 3:
        frames.append([[round(float(v), 3) for v in q] for q in d.qpos.numpy()[show, 3:7]])
wp.synchronize()

ke1 = kinetic_energy(d.qvel.numpy())
print(f"{'world':>5} {'wobble [rad/s]':>15} {'first flip [s]':>15} {'KE drift (RK4)':>15}")
for w in show:
    print(f"{w:>5} {wobble[w]:>15.5f} {flip_t[w]:>15.3f} {(ke1[w] - ke0[w]) / ke0[w]:>+15.1e}")

# the same world 0 with the default Euler integrator, in C MuJoCo (fast for one world)
mjm_euler = mujoco.MjModel.from_xml_string(RACKET.replace(' integrator="RK4"', ""))
mjd = mujoco.MjData(mjm_euler)
mjd.qvel[:] = qvel[0]
for _ in range(NSTEP):
    mujoco.mj_step(mjm_euler, mjd)
ke_e = kinetic_energy(mjd.qvel[None, :])[0]
print(f"world 0 with Euler instead (C MuJoCo, same dt): KE drift {(ke_e - ke0[0]) / ke0[0]:+.1e}")

mjw.forward(m, d)                                   # recompute qfrc_bias for the current qvel
w0 = d.qvel.numpy()[0, 3:6].astype(np.float64)
gyro = np.cross(w0, I * w0)
print("world 0: qfrc_bias[3:6] =", d.qfrc_bias.numpy()[0, 3:6])
print("world 0: w x (I w)      =", gyro)

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": 8, "dt": 4 * float(mjm.opt.timestep), "kind": "quat",
                   "half": [0.05, 0.2, 0.4], "labels": [f"world {w}" for w in show],
                   "wobble": [round(float(wobble[w]), 5) for w in show],
                   "flip": [round(float(flip_t[w]), 3) for w in show],
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## §7 Joints and reduced coordinates
A pendulum in maximal coordinates (x, z + a stiff spring rod) vs one hinge angle. Stiffer rods drift less but need smaller steps; the hinge is exact by construction.

In [ ]:
# Example 3.8 - ex8_pendulum_two_ways.py
"""Example 3.8 - one pendulum, two descriptions: maximal coordinates vs reduced coordinates.

(a) Maximal: a point mass with free (x, z) and a very stiff spring standing in for the rod,
    in NumPy with semi-implicit Euler. 2 coordinates, and the rod length is only kept
    approximately; stiffer rods drift less but need smaller time steps.
(b) Reduced: a hinge joint in MuJoCo. 1 coordinate (the angle); the rod length is exact
    by construction.
We print the period from both, the exact large-amplitude period, and the rod-length drift of (a).
Run: python ex8_pendulum_two_ways.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

L, G, TH0 = 0.5, 9.81, 0.5          # rod length [m], gravity, start angle [rad] from straight down
DT, T = 0.001, 5.0

PENDULUM_HINGE = f"""
<mujoco model="hinge_pendulum">
  <option timestep="{DT}"/>
  <worldbody>
    <body name="pivot">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="sphere" pos="0 0 -{L}" size=".01" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""


def period_from(x, dt):
    """Mean time between upward zero crossings of x(t)."""
    up = np.nonzero((x[:-1] < 0) & (x[1:] >= 0))[0]
    # linear interpolation of the crossing time inside the step
    t = (up + x[up] / (x[up] - x[up + 1])) * dt
    return float(np.diff(t).mean())


def exact_period(L, g, th0):
    """T = 4 sqrt(L/g) K(sin(th0/2)); K by the arithmetic-geometric mean."""
    a, b = 1.0, np.sqrt(1 - np.sin(th0 / 2) ** 2)
    for _ in range(30):
        a, b = (a + b) / 2, np.sqrt(a * b)
    return 4 * np.sqrt(L / g) * np.pi / (2 * a)


def cartesian_pendulum(k):
    """(a) point mass on a stiff spring 'rod' (k per unit mass), semi-implicit Euler."""
    p = L * np.array([np.sin(TH0), -np.cos(TH0)])     # (x, z), pivot at the origin
    v = np.zeros(2)
    xs, stretch = [], 0.0
    for _ in range(round(T / DT)):
        r = np.linalg.norm(p)
        a = np.array([0.0, -G]) - k * (r - L) * p / r  # gravity + spring along the rod
        v = v + a * DT
        p = p + v * DT
        xs.append(p[0])
        stretch = max(stretch, abs(np.linalg.norm(p) - L))
        if not np.isfinite(p).all() or stretch > 10 * L:
            return None, np.inf
    return period_from(np.array(xs), DT), stretch


print(f"L = {L} m, start angle {TH0} rad, dt = {DT} s, {T} s simulated")
print(f"small-angle period 2 pi sqrt(L/g) = {2 * np.pi * np.sqrt(L / G):.4f} s")
print(f"exact period for this amplitude   = {exact_period(L, G, TH0):.4f} s")
print("(a) maximal coordinates (x, z) + stiff spring rod, 2 DOF:")
for k in [1e3, 1e4, 1e5, 1e6, 1e7]:
    per, st = cartesian_pendulum(k)
    if per is None:
        print(f"    k/m = {k:8.0e}: unstable (sqrt(k/m)*dt = {np.sqrt(k) * DT:.2f} > 2), blew up")
    else:
        print(f"    k/m = {k:8.0e}: period = {per:.4f} s   "
              f"max rod-length error = {1000 * st:7.3f} mm")

mjm = mujoco.MjModel.from_xml_string(PENDULUM_HINGE)
mjd = mujoco.MjData(mjm)
mjd.qpos[0] = TH0
xs = []
for _ in range(round(T / DT)):
    mujoco.mj_step(mjm, mjd)
    xs.append(mjd.geom_xpos[0, 0])
rod = np.linalg.norm(mjd.geom_xpos[0] - mjd.xpos[1])
print(f"(b) reduced coordinate (hinge angle), nq = {mjm.nq}, nv = {mjm.nv}:")
print(f"    period = {period_from(np.array(xs), DT):.4f} s   rod length at the end = {rod:.6f} m")

## §8 Constraints: a resting contact cancels gravity
Contacts and joint limits are one-sided, equalities two-sided. All are soft: the resting ball sits 0.367 mm inside the floor, which MuJoCo's reference acceleration aref = -k*imp*pos - b*vel predicts.

In [ ]:
# Example 3.9 - ex9_contact_as_constraint.py
"""Example 3.9 - a resting contact is a constraint force that cancels gravity.

BALL_DROP in MJWarp: each world drops the ball from a different height and lets it settle for
1 s. Then, per world, we print the bias force on the vertical dof (gravity, qfrc_bias[2]), the
contact normal force decoded by mjw.contact_force, their balance, the remaining vertical
acceleration, and the small penetration contact.dist that the soft contact needs (Ch 9).
Run: python ex9_contact_as_constraint.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 500                                   # 1 s: enough for every world to come to rest

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = np.linspace(0.3, 0.8, NWORLD)    # different drop heights per world
d.qpos.assign(qpos)

for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

nacon = int(d.nacon.numpy()[0])
force = wp.zeros(nacon, dtype=wp.spatial_vector)
ids = wp.array(np.arange(nacon, dtype=np.int32), dtype=int)
mjw.contact_force(m, d, ids, False, force)       # contact frame: entry 0 is the normal force

f_n = force.numpy()[:, 0]
wid = d.contact.worldid.numpy()[:nacon]
dist = d.contact.dist.numpy()[:nacon]
bias = d.qfrc_bias.numpy()[:, 2]
qacc = d.qacc.numpy()[:, 2]
nefc = d.nefc.numpy()

print(f"nworld = {NWORLD}, contacts in the shared pool: nacon = {nacon}, "
      f"mg = {mjm.body_mass[1] * 9.81:.4f} N")
print(f"{'world':>5} {'qfrc_bias[2]':>13} {'normal force':>13} {'balance':>9} "
      f"{'qacc[2]':>9} {'dist [mm]':>10} {'nefc':>5}")
for c in range(min(nacon, 4)):
    w = wid[c]
    print(f"{w:>5} {bias[w]:>13.4f} {f_n[c]:>13.4f} {f_n[c] - bias[w]:>+9.4f} "
          f"{qacc[w]:>+9.4f} {1000 * dist[c]:>10.3f} {nefc[w]:>5}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## §9 Bridge to MuJoCo: M(q) qacc + c(q, qvel) = tau + J^T f
You derive M and c in Ch 11. Here, just check that the equation balances on the double pendulum.

In [ ]:
# Example 3.10 - ex10_preview_eom.py
"""Example 3.10 - a preview of the equation of motion M(q) qacc + c(q, qvel) = tau.

The double PENDULUM from Ch 2, upper link horizontal (keyframe "horizontal"), elbow bent by
0.6 rad, both joints moving, with 1 N m on the shoulder motor. C MuJoCo's mj_forward fills every
term; we print them and the residual, which is ~0 because this scene has no contacts or other
constraints. Then MJWarp's mjw.forward computes qacc for the same state. You derive all of this
in Ch 11.
Run: python ex10_preview_eom.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjd = mujoco.MjData(mjm)
mujoco.mj_resetDataKeyframe(mjm, mjd, mjm.key("horizontal").id)
mjd.qpos[1] = 0.6                  # bend the elbow
mjd.qvel[:] = [1.0, 0.5]           # moving, so the velocity-dependent part of c is not zero
mjd.ctrl[0] = 1.0                  # 1 N m on the shoulder
mujoco.mj_forward(mjm, mjd)

M = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, M)       # dense copy of the (sparse) mass matrix
c = mjd.qfrc_bias                  # gravity + Coriolis/centrifugal ("bias") forces
tau = mjd.qfrc_actuator + mjd.qfrc_passive + mjd.qfrc_applied
print("M (mass matrix, depends on q):")
print(M)
still = mujoco.MjData(mjm)         # same pose, zero velocity: c(q, 0) is gravity alone
still.qpos[:] = mjd.qpos
mujoco.mj_forward(mjm, still)
print("c   = qfrc_bias     :", c, " = gravity", still.qfrc_bias,
      "+ velocity part", c - still.qfrc_bias)
print("tau = qfrc_actuator :", mjd.qfrc_actuator, " (+ passive", mjd.qfrc_passive, ")")
print("qacc                :", mjd.qacc)
print("residual M qacc + c - tau:", M @ mjd.qacc + c - tau, " (no constraints: J^T f = 0)")
print("qacc = M^-1 (tau - c):", np.linalg.solve(M, tau - c))

# MJWarp computes the same terms on the device
m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd)
mjw.forward(m, d)
print("MJWarp qacc (float32):", d.qacc.numpy()[0])
print("MJWarp qfrc_bias     :", d.qfrc_bias.numpy()[0])

## Your own physics engine in 40 lines
Everything above in one class: a batched state (one ball per world), a forward pass (gravity, collision detection, a one-sided soft floor) and a semi-implicit integrator.

In [ ]:
# Your own physics engine in ~40 lines: many worlds, one ball each, a soft floor.
import numpy as np


class TinyEngine:
    """Batched point-mass engine: state (z, v) per world, semi-implicit Euler, penalty floor."""

    def __init__(self, nworld, radius=0.1, dt=0.002, timeconst=0.02, dampratio=1.0, g=9.81):
        self.nworld, self.radius, self.dt, self.g = nworld, radius, dt, g
        self.k = 1.0 / (timeconst * dampratio) ** 2     # floor stiffness per unit mass
        self.b = 2.0 / timeconst                         # floor damping per unit mass
        self.z = np.ones(nworld)                         # qpos: one number per world
        self.v = np.zeros(nworld)                        # qvel
        self.t = 0.0

    def forces(self):
        """Acceleration of every world at the current state (the 'forward' pass)."""
        a = np.full(self.nworld, -self.g)                # gravity
        pen = self.radius - self.z                       # > 0 inside the floor
        contact = pen > 0                                # 'collision detection'
        push = np.maximum(0.0, self.k * pen - self.b * self.v)
        a[contact] += push[contact]                      # one-sided 'constraint' force
        return a

    def step(self):
        """One time step: forward, then integrate (velocity first, like MuJoCo's Euler)."""
        a = self.forces()
        self.v += a * self.dt
        self.z += self.v * self.dt
        self.t += self.dt


eng = TinyEngine(nworld=4)
eng.z[:] = np.linspace(0.3, 0.8, 4)
for _ in range(500):
    eng.step()
print(f"t = {eng.t:.2f} s, z per world:", np.round(eng.z, 4))
print("rest penetration [mm]:", np.round(1000 * (eng.radius - eng.z), 3))

## Exercise 3.1: a projectile with air drag (auto-checked)
A ball leaves the ground at 30 m/s and 45 degrees. Drag adds -(c/m)|v| v with c/m = 0.01 1/m. Write `landing_distance(dt)` with semi-implicit Euler, interpolate the landing point inside the last step, and check that dt = 0.01 lands within 2 percent of a dt = 1e-5 reference.

In [ ]:
import numpy as np

G = np.array([0.0, -9.81])
C_OVER_M = 0.01


def landing_distance(dt, v0=30.0, angle_deg=45.0):
    ...  # your semi-implicit Euler loop; return x at z = 0


ref = landing_distance(1e-5)
mine = landing_distance(0.01)
if mine is None:
    print("fill in landing_distance")
else:
    assert abs(mine - ref) / ref < 0.02

In [ ]:
#@title Exercise 3.1: solution (double-click to show the code)
import numpy as np

G = np.array([0.0, -9.81])
C_OVER_M = 0.01            # quadratic drag coefficient per unit mass [1/m]


def landing_distance(dt, v0=30.0, angle_deg=45.0):
    """Semi-implicit Euler for a projectile with drag a = g - (c/m)|v| v. Returns x at z = 0."""
    a0 = np.radians(angle_deg)
    p = np.zeros(2)
    v = v0 * np.array([np.cos(a0), np.sin(a0)])
    while True:
        a = G - C_OVER_M * np.linalg.norm(v) * v
        v = v + a * dt
        p_new = p + v * dt
        if p_new[1] < 0:                        # crossed the ground in this step
            s = p[1] / (p[1] - p_new[1])        # linear interpolation inside the step
            return p[0] + s * (p_new[0] - p[0])
        p = p_new


ref = landing_distance(1e-5)
mine = landing_distance(0.01)
print(f"reference (dt=1e-5): {ref:.3f} m, dt=0.01: {mine:.3f} m, "
      f"error {100 * abs(mine - ref) / ref:.3f} %")
print(f"no drag would give v0^2/g = {30.0**2 / 9.81:.3f} m")
assert abs(mine - ref) / ref < 0.02

## Exercise 3.2: the inertia of a cylinder (auto-checked)
A cylinder geom of radius 0.1 m, half-length 0.3 m, mass 2 kg (MJCF `size="radius half_length"`). Compute its principal moments by hand and assert them against `mjm.body_inertia[1]`.

In [ ]:
import mujoco
import numpy as np

r, h, m = 0.1, 0.3, 2.0
xml = ...          # a body with a freejoint and the cylinder geom
I_mine = ...       # [I_xx, I_yy, I_zz] about the centre; the cylinder axis is z
if xml is ... or I_mine is ...:
    print("fill in xml and I_mine")
else:
    mjm = mujoco.MjModel.from_xml_string(xml)
    assert np.allclose(I_mine, mjm.body_inertia[1], rtol=1e-6)

In [ ]:
#@title Exercise 3.2: solution (double-click to show the code)
import mujoco
import numpy as np

r, h, m = 0.1, 0.3, 2.0          # radius, HALF length, mass
xml = f"""<mujoco><worldbody><body><freejoint/>
  <geom type="cylinder" size="{r} {h}" mass="{m}"/></body></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(xml)

I_axial = m * r**2 / 2                          # about the cylinder axis (z)
I_side = m * (3 * r**2 + (2 * h) ** 2) / 12     # about x and y through the centre
I_mine = np.array([I_side, I_side, I_axial])
print("formula:", I_mine, " MuJoCo:", mjm.body_inertia[1])
assert np.allclose(I_mine, mjm.body_inertia[1], rtol=1e-6)

## Stretch: two balls that collide
Extend `TinyEngine` to 2-D positions and two balls per world, with a ball-ball penalty contact along the line between the centres (equal and opposite forces, damper on the relative normal velocity). Check that momentum is conserved with gravity and the floor switched off, then compare with the same scene in MJCF + MJWarp.

In [ ]:
# your stretch solution here

## Last cell: TinyEngine vs MJWarp
Same four drops in both engines. In free flight they agree to every digit (same gravity, same semi-implicit Euler, same 0.002 s step); at the floor a penalty spring and MuJoCo's soft constraint part ways (3.92 mm vs 0.367 mm rest depth). Run the TinyEngine cell above first.

In [ ]:
# Compare TinyEngine with MJWarp on the same four drops (run the TinyEngine cell first).
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL = """<mujoco><option timestep="0.002"/><worldbody><geom type="plane" size="5 5 .1"/>
<body pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1"/></body></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(BALL)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=4, njmax=16)
qpos = np.tile(mjm.qpos0, (4, 1)).astype(np.float32)
qpos[:, 2] = np.linspace(0.3, 0.8, 4)
d.qpos.assign(qpos)

mine = TinyEngine(nworld=4)
mine.z[:] = qpos[:, 2]
print(" t [s]   world 3 z: TinyEngine   MJWarp")
for i in range(500):
    mjw.step(m, d)
    mine.step()
    if i % 50 == 49:
        print(f" {(i + 1) * 0.002:.1f}               {mine.z[3]:.4f}     {d.qpos.numpy()[3, 2]:.4f}")
print("rest penetration [mm]: TinyEngine", round(1000 * (0.1 - mine.z[0]), 3),
      " MJWarp", round(1000 * (0.1 - float(d.qpos.numpy()[0, 2])), 3))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())